In [1]:
from database.adatabase import ADatabase
import pandas as pd
from processor.processor import Processor as p
from xgboost import XGBRegressor

## run this code on a monthly basis
fed = ADatabase("fed")

In [2]:
fed.connect()
cpi = p.column_date_processing(fed.retrieve("cpi").rename(columns={"value":"cpi"}))
exports = p.column_date_processing(fed.retrieve("exports").rename(columns={"value":"exports"}))
gdp = p.column_date_processing(fed.retrieve("gdp").rename(columns={"value":"gdp"}))
unrate = p.column_date_processing(fed.retrieve("unrate").rename(columns={"value":"unrate"}))
treasury_yields = p.column_date_processing(fed.retrieve("treasury_yields").rename(columns={"value":"treasury_yields"}))
oil = p.column_date_processing(fed.retrieve("oil").rename(columns={"value":"oil"}))
sp500 = p.column_date_processing(fed.retrieve("sp500").rename(columns={"value":"sp500"}))
fed.disconnect()
factor_sets = [cpi,exports,gdp,unrate,oil,treasury_yields]
base = sp500
for factor_set in factor_sets:
    base = p.merge(base,factor_set,on="date")
base = base.ffill().bfill().dropna()
print(base.columns)
factors = ["cpi","exports","gdp","unrate","oil","sp500"]
base.to_csv("us_macro.csv")
model_data = p.merge(sp500,base,on="date").dropna()
model_data = model_data.drop(["date","realtime_start","realtime_end"],axis=1).groupby(["year","quarter"]).first().reset_index()
model_data = model_data.apply(pd.to_numeric)
model_data.sort_values(["year","quarter"],inplace=True)
model_data["y"] = model_data["sp500"].shift(-4)

Index(['date', 'sp500', 'year', 'quarter', 'month', 'week', 'weekday',
       'realtime_start', 'realtime_end', 'cpi', 'exports', 'gdp', 'unrate',
       'oil'],
      dtype='object')


In [3]:
training_data = model_data[model_data["year"]<2021].dropna().copy().reset_index()
recommendation_data = model_data[model_data["year"]>=2021].copy().reset_index()

In [5]:
training_data.tail()

,index,year,quarter,sp500,month,week,weekday,cpi,exports,gdp,unrate,oil,y
24,24,2019,4,2940.25,10,40,1,257.244,-516.146,21902.390,3.6,53.60,3380.80
25,25,2020,1,3257.85,1,1,3,257.803,-516.146,21902.390,3.6,61.17,3700.65
26,26,2020,2,2470.50,4,14,2,256.092,-531.137,19913.143,14.7,20.28,4019.87
27,27,2020,3,3115.86,7,27,2,258.278,-695.747,21647.640,10.2,39.88,4319.94
28,28,2020,4,3380.80,10,40,3,260.286,-760.659,22024.502,6.9,38.51,4357.04


In [6]:
recommendation_data.tail()

,index,year,quarter,sp500,month,week,weekday,cpi,exports,gdp,unrate,oil,y
7,36,2022,4,3678.43,10,40,0,296.539,-892.026,25994.639,3.5,84.05,4288.39
8,37,2023,1,3824.14,1,1,1,298.990,-892.026,25994.639,3.5,76.87,NaN
9,38,2023,2,4124.51,4,14,0,301.808,-892.026,25994.639,3.5,80.40,NaN
10,39,2023,3,4455.59,7,27,0,303.841,-892.026,25994.639,3.6,69.71,NaN
11,40,2023,4,4288.39,10,40,0,307.481,-892.026,25994.639,3.8,88.81,NaN


In [7]:
model = XGBRegressor(booster="dart",learning_rate=1)
model.fit(training_data[factors],training_data["y"])

XGBRegressor(base_score=None, booster='dart', callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [8]:
recommendation_data["prediction"] = model.predict(recommendation_data[factors])

In [9]:
recommendation_data

,index,year,quarter,sp500,month,week,weekday,cpi,exports,gdp,unrate,oil,y,prediction
0,29,2021,1,3700.65,1,1,0,262.035,-760.659,22024.502,6.7,47.47,4796.56,4150.338867
1,30,2021,2,4019.87,4,13,3,266.670,-831.972,23292.362,6.1,61.41,4545.86,3888.479004
2,31,2021,3,4319.94,7,26,3,271.764,-884.277,23828.973,5.4,75.33,3825.33,3888.477783
3,32,2021,4,4357.04,10,39,4,276.522,-924.289,24654.603,4.5,76.01,3678.43,3887.441895
4,33,2022,1,4796.56,1,1,0,280.887,-924.289,24654.603,3.9,75.99,3824.14,3887.441895
5,34,2022,2,4545.86,4,13,4,288.611,-1025.567,25544.273,3.6,99.32,4124.51,3865.803467
6,35,2022,3,3825.33,7,26,4,294.628,-892.026,25994.639,3.5,110.30,4455.59,3865.803467
7,36,2022,4,3678.43,10,40,0,296.539,-892.026,25994.639,3.5,84.05,4288.39,3865.803467
8,37,2023,1,3824.14,1,1,1,298.990,-892.026,25994.639,3.5,76.87,NaN,3865.803467
9,38,2023,2,4124.51,4,14,0,301.808,-892.026,25994.639,3.5,80.40,NaN,3865.803467


In [10]:
## note the format of this db is year quarter prediction
fed.connect()
fed.drop("sp500_v2_projections")
fed.store("sp500_v2_projections",recommendation_data[["year","quarter","prediction"]])
fed.disconnect()